# MLOps Assignment 2 - DistilBERT Goodreads Genre Classification

### Complete unified solution (Kaggle / Colab / local)

Fine-tune DistilBERT on UCSD Goodreads reviews to classify them into 8 book genres, track the run with Weights & Biases, and publish the model to the Hugging Face Hub.

In [1]:
!pip install -q -U transformers wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 71.4 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

In [3]:
import gzip
import json
import random

import requests
import torch
import wandb
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (DistilBertForSequenceClassification,
                          DistilBertTokenizerFast, Trainer, TrainingArguments)

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 512
SEED = 42
GENRES = ["children", "comics_graphic", "fantasy_paranormal", "history_biography",
          "mystery_thriller_crime", "poetry", "romance", "young_adult"]
NUM_LABELS = len(GENRES)
GENRE_URL_TEMPLATE = ("https://mcauleylab.ucsd.edu/public_datasets/gdrive/"
                      "goodreads/byGenre/goodreads_reviews_{genre}.json.gz")
SAMPLE_SIZE = 1000   # reviews per genre; lower to 200 if GPU hours run low
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [4]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nikethv6 (nikethv6-iitj) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
!pip uninstall -y transformers huggingface_hub
!pip install transformers==4.51.3 huggingface_hub==0.30.2 -q

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: huggingface_hub 1.10.1
Uninstalling huggingface_hub-1.10.1:
  Successfully uninstalled huggingface_hub-1.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


In [6]:
def load_reviews(url, head=10000, sample_size=SAMPLE_SIZE):
    """Stream a gzipped genre file and return a random subset of review texts."""
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    reviews = []
    with gzip.open(response.raw, "rt", encoding="utf-8") as handle:
        for count, line in enumerate(handle):
            if count >= head:
                break
            try:
                text = json.loads(line).get("review_text", "").strip()
            except json.JSONDecodeError:
                continue
            if text:
                reviews.append(text)
    return random.sample(reviews, min(sample_size, len(reviews)))


random.seed(SEED)
train_texts, train_labels, test_texts, test_labels = [], [], [], []
train_per_genre = int(SAMPLE_SIZE * 0.8)
for genre in GENRES:
    print("Loading genre:", genre)
    reviews = load_reviews(GENRE_URL_TEMPLATE.format(genre=genre))
    for review in reviews[:train_per_genre]:
        train_texts.append(review)
        train_labels.append(genre)
    for review in reviews[train_per_genre:]:
        test_texts.append(review)
        test_labels.append(genre)
print(f"{len(train_texts)} train / {len(test_texts)} test reviews")

Loading genre: children
Loading genre: comics_graphic
Loading genre: fantasy_paranormal
Loading genre: history_biography
Loading genre: mystery_thriller_crime
Loading genre: poetry
Loading genre: romance
Loading genre: young_adult
6400 train / 1600 test reviews


In [7]:
label2id = {genre: idx for idx, genre in enumerate(GENRES)}
id2label = {idx: genre for genre, idx in label2id.items()}

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_enc = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LENGTH)


class MyDataset(torch.utils.data.Dataset):
    """Wrap tokenizer encodings + integer labels for the HuggingFace Trainer."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = MyDataset(train_enc, [label2id[label] for label in train_labels])
test_dataset = MyDataset(test_enc, [label2id[label] for label in test_labels])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)
print(model.config.num_labels, "output labels")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


8 output labels


In [9]:
wandb.init(
    project="mlops-assignment2",
    name="distilbert-run-1",
    config={
        "model": MODEL_NAME,
        "epochs": 5,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "max_length": MAX_LENGTH,
        "platform": "Kaggle"
    }
)

wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260527_184923-jek60kbf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert-run-1
wandb: ⭐️ View project at https://wandb.ai/nikethv6-iitj/mlops-assignment2
wandb: 🚀 View run at https://wandb.ai/nikethv6-iitj/mlops-assignment2/runs/jek60kbf


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="distilbert-run-1"
)

In [11]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.619031,2.544293,0.540000,0.550221
2,2.029818,2.327427,0.581250,0.580518
3,1.425848,2.447194,0.586250,0.590011
4,0.939315,2.623217,0.593750,0.594752
5,0.619086,2.755960,0.586250,0.587467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=1.6980481224060058, metrics={'train_runtime': 973.9815, 'train_samples_per_second': 32.855, 'train_steps_per_second': 1.027, 'total_flos': 4239410331648000.0, 'train_loss': 1.6980481224060058, 'epoch': 5.0})

In [14]:
eval_results = trainer.evaluate()

print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.619086,2.327427,5,0.581250,0.580518


{'eval_loss': 2.3274266719818115, 'eval_accuracy': 0.58125, 'eval_f1': 0.5805178305491367}


In [15]:
preds = trainer.predict(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [16]:
import json
from sklearn.metrics import classification_report

pred_labels = preds.predictions.argmax(-1)

true_labels = [item["labels"].item() for item in test_dataset]

report = classification_report(
    true_labels,
    pred_labels,
    target_names=list(id2label.values()),
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

In [17]:
artifact = wandb.Artifact(
    "eval-report",
    type="evaluation"
)

artifact.add_file("eval_report.json")

wandb.log_artifact(artifact)

<Artifact eval-report>

In [18]:
from huggingface_hub import login

login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [19]:
model.push_to_hub(
    "Niketh0503/distilbert-goodreads-genres"
)

tokenizer.push_to_hub(
    "Niketh0503/distilbert-goodreads-genres"
)

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ModuleNotFoundError: No module named 'huggingface_hub.utils._xet_progress_reporting'

In [ ]:
wandb.run.summary["huggingface_model"] = \
"https://huggingface.co/Niketh0503/distilbert-goodreads-genres"

In [ ]:
wandb.finish()